In [1]:
#Load Project Environment
import Pkg
Pkg.activate(dirname(@__DIR__))
Pkg.instantiate()

  Activating project at `c:\Users\pbb62\Documents\Repositories\CHANCE_C.jl`
Precompiling project...
  ✓ CHANCE_C
  1 dependency successfully precompiled in 12 seconds. 315 already precompiled.


In [2]:
#Load Packages
using CSV, DataFrames
using DataStructures
using Agents
using Statistics,StatsBase,Distributions
using CategoricalArrays

include(joinpath(dirname(@__DIR__), "src/CHANCE_C.jl"))
using .CHANCE_C

In [206]:
###Load Input data:
##For flood history input
f_df = DataFrame(CSV.File(joinpath(dirname(@__DIR__), "data", "synth_flood_phil.csv")))

##For BG
#open bg file
phil_bg = DataFrame(CSV.File(joinpath(dirname(@__DIR__), "data/phil_flood_bg_2019.csv")))

##load pop data
phil_cbsa_base_pop = DataFrame(CSV.File(joinpath(dirname(dirname(@__DIR__)), "philadelphia-data/model_inputs/pop_files/philly_cbsa_pop_0.csv")))
#drop missing values
dropmissing!(phil_cbsa_base_pop, :NP)
#For rows with people and negative income, set income to bottom 10%
inc_bot_10 = quantile(subset(phil_cbsa_base_pop, [:NP .=> ByRow(>(0)), :adj_income_2019 .=> ByRow(>(0))]).adj_income_2019, [0.10])[1]
@. phil_cbsa_base_pop.adj_income_2019 = ifelse.(phil_cbsa_base_pop.NP > 0 && phil_cbsa_base_pop.adj_income_2019 <= 0, inc_bot_10, phil_cbsa_base_pop.adj_income_2019)

#Subset to Phil. County (Not part of function)
phil_base_pop = subset(phil_cbsa_base_pop, :county => x -> x .== 42101)

Row,serialno,year,state,puma,rep,county,tract,bg,puma10,GEOID,RAC1P,NP,HINCP,ADJINC,adj_income_2019
,String15,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64?,Float64,Float64?,Float64?,Float64
1,2015000000403,2015,42,3201,0,42101,35100,1,4203201,421010351001,1.0,3.0,100000.0,1.08047,108047.0
2,2015000000403,2015,42,3201,0,42101,35200,1,4203201,421010352001,1.0,3.0,100000.0,1.08047,108047.0
3,2015000000403,2015,42,3201,0,42101,35500,3,4203201,421010355003,1.0,3.0,100000.0,1.08047,108047.0
4,2015000000403,2015,42,3201,0,42101,35500,3,4203201,421010355003,1.0,3.0,100000.0,1.08047,108047.0
5,2015000000403,2015,42,3201,0,42101,36100,1,4203201,421010361001,1.0,3.0,100000.0,1.08047,108047.0
6,2015000000403,2015,42,3201,0,42101,36201,3,4203201,421010362013,1.0,3.0,100000.0,1.08047,108047.0
7,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0
8,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0
9,2015000000403,2015,42,3201,0,42101,36202,3,4203201,421010362023,1.0,3.0,100000.0,1.08047,108047.0


In [4]:
#Read in flood data
phil_flood = DataFrame(CSV.File(joinpath(dirname(dirname(@__DIR__)), "philadelphia-data", "model_inputs", "phil_flood_hist_year.csv")))
#transform df to correct format
phil_flood_rec = unstack(phil_flood, :GEOID, :year, :perc_flood_extent)
#Extra edits
phil_flood_rec[!,"1982"] = zeros(size(phil_flood_rec)[1])
select!(phil_flood_rec, "GEOID", "1981", "1982", Not(["1982", "2019"]), "2019")

Row,GEOID,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
,Int64,Float64?,Float64,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?,Float64?
1,421010001001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,421010001002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,421010002001,0.00214058,0.0,0.0,0.00107029,0.0,0.00267571,0.000891913,0.0,0.00552983,0.0,0.00410277,0.0,0.00249733,0.0,0.00178382,0.00499468,0.0,0.00410277,0.00463793,0.0,0.0019622,0.000891913,0.0,0.00231898,0.00249733,0.0,0.00338924,0.00356763,0.0,0.0,0.00499468,0.0,0.0,0.00107029,0.0,0.0,0.00356763,0.00303249,0.00249733
4,421010003001,0.0496589,0.0,0.0,0.0390932,0.0,0.0669868,0.03698,0.0,0.186591,0.0,0.11897,0.0,0.0739602,0.0,0.0475458,0.158275,0.0,0.11728,0.182788,0.0,0.049025,0.0386705,0.0,0.0853713,0.0792431,0.0,0.091922,0.123619,0.0,0.0,0.188282,0.0,0.0,0.0452213,0.0,0.0,0.10439,0.0860052,0.0636058
5,421010003002,0.0348188,0.0,0.0821608,0.164015,0.0323759,0.044287,0.115757,0.00855201,0.20311,0.00977373,0.0629182,0.0109954,0.0448978,0.0109954,0.102013,0.177759,0.00855201,0.0620019,0.318561,0.0256564,0.0366513,0.0253504,0.0125226,0.22785,0.0455087,0.226017,0.0739141,0.059253,0.0109954,0.216854,0.318255,0.0198531,0.01069,0.0418438,0.0103846,0.0100792,0.0574205,0.0421491,0.0366513
6,421010004011,0.0273739,0.0,0.0333518,0.0481398,0.012271,0.0390157,0.0503425,0.00755141,0.09219,0.00818069,0.0493991,0.0113271,0.0377572,0.0119564,0.0506571,0.091561,0.00755141,0.0487698,0.162985,0.012271,0.0330375,0.0235982,0.012271,0.0676475,0.0431061,0.0682768,0.0446793,0.0493991,0.0106978,0.0629278,0.162041,0.012271,0.012271,0.0320936,0.00912462,0.00818069,0.045938,0.0374426,0.031779
7,421010004021,0.0227354,0.0,0.0,0.0227354,0.0,0.0341032,0.0211115,0.0,0.115303,0.0,0.0698309,0.0,0.0357272,0.0,0.0276073,0.0925669,0.0,0.0698309,0.108807,0.0,0.0243594,0.0227354,0.0,0.0227354,0.0357272,0.0,0.0357272,0.0763269,0.0,0.0,0.110431,0.0,0.0,0.0227354,0.0,0.0,0.0600869,0.0357272,0.0292313
8,421010004022,0.0153833,0.0,0.0,0.011733,0.0,0.0198159,0.0114723,0.0,0.0461502,0.0,0.0284201,0.0,0.0219017,0.0,0.0151226,0.0391103,0.0,0.0284201,0.0417177,0.0,0.0151226,0.011733,0.0,0.0213803,0.0211195,0.0,0.0245091,0.0278987,0.0,0.0,0.0435428,0.0,0.0,0.0138189,0.0,0.0,0.0268557,0.0224232,0.0185122
9,421010005001,0.00254584,0.0,0.0,0.00159116,0.0,0.0031823,0.00159116,0.0,0.00906949,0.0,0.00461434,0.0,0.00318229,0.0,0.00222761,0.00700102,0.0,0.00461434,0.0105014,0.0,0.00254584,0.00175027,0.0,0.00254584,0.00302318,0.0,0.00381876,0.00429611,0.0,0.0,0.0103423,0.0,0.0,0.0020685,0.0,0.0,0.00429611,0.00334142,0.00302318


In [208]:
#Define input Parameters
no_of_years = 39
start_year = 1981
no_hhs_per_agent=10
growth_rate = 0.01
grouped = true
group_col = "adj_income_2019"
cutoff_dict = OrderedDict(1 => [-60000.00,25000.00], 2 =>[25000.00,75000.00], 3 =>[75000.00, 1e7]) #1=> "low income", 2=> "medium income", 3=> "high income"
bg_cat = Dict(:col =>"income_cat", :group => [1,2,3])
simple_anova_coefficients = Dict(1=> [0, 294707, 130553, 128990, 154887, 72443], 2=> [0, 294707, 130553, 128990, 154887, 72443], 3=> [0, 294707, 130553, 128990, 154887, 72443])
house_budget_mode = "rhea"
house_choice_mode = "flood_mem_utility"
risk_averse = 0.3
base_move = 0.01
flood_mem = 10
seed = 1500

1500

In [8]:
### Calculate Flood matrix and Dict for ABM input
f_matrix, f_dict = CHANCE_C.flood_history(f_df; no_of_years = no_of_years, start_year = start_year)

([0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0;;;], Dict(5 => (1, 5), 16 => (1, 16), 20 => (1, 20), 35 => (1, 35), 12 => (1, 12), 24 => (1, 24), 28 => (1, 28), 8 => (1, 8), 17 => (1, 17), 30 => (1, 30)…))

In [ ]:
#Try with actual flood history
phil_f_matrix, phil_f_dict = CHANCE_C.flood_history(phil_flood_rec; no_of_years = no_of_years, start_year = start_year)


In [209]:
### Initialize ABM
phil_abm = CHANCE_C.Simulator(phil_bg, phil_base_pop, f_matrix, f_dict, CHANCE_C.model_step!; no_of_years = no_of_years, no_hhs_per_agent = no_hhs_per_agent,
house_budget_mode = house_budget_mode, house_choice_mode = house_choice_mode, grouped = grouped, group_col = group_col, cutoff_dict = cutoff_dict, bg_cat = bg_cat,
risk_averse = risk_averse, flood_mem = flood_mem, perc_move = base_move, seed = seed)

StandardABM with 53664 agents of type Union{BlockGroup, HHAgent, Main.CHANCE_C.Queue}
 agents container: Dict
 space: GridSpace with size (28, 28), metric=chebyshev, periodic=true
 scheduler: Agents.Schedulers.ByType
 properties: df, total_population, flood_hazard, agent_creation, relo_sampler, agent_relocate, build_develop, house_price, hh_utilities_df, no_of_years, flood_matrix, flood_dict, tick

In [210]:
init_pop = length([a for a in allagents(phil_abm) if a isa HHAgent && a.bg_id >= 1])

35820

In [211]:
migrant_ids = [a.id for a in agents_in_position(phil_abm[-1].pos, phil_abm) if a isa CHANCE_C.HHAgent]
no_new_agents = floor(Int64, (length([a for a in allagents(phil_abm) if a isa CHANCE_C.HHAgent]) - length(migrant_ids)) * growth_rate)

358

In [212]:
phil_abm.tick += 1
AgentMigration(phil_abm; growth_rate = 0.01)

In [213]:
length([a for a in allagents(phil_abm) if a isa HHAgent && a.bg_id >= 1])

35820

In [214]:
for id in collect(Agents.schedule(phil_abm))
    if phil_abm[id] isa CHANCE_C.Queue || (phil_abm[id] isa HHAgent && phil_abm[id].bg_id < 1) #Dont involve HHAgents in Queues
        continue
    else
        CHANCE_C.agent_step!(phil_abm[id],phil_abm)
    end
end

In [215]:
length([a for a in allagents(phil_abm) if a isa HHAgent && a.bg_id >= 1])

35439

In [216]:
queue_1 = collect(agents_in_position(phil_abm[0], phil_abm))
ag_in_queue_1 = length(queue_1) - 1

737

In [164]:
ag_in_queue_1/(init_pop + no_new_agents)

0.020371496489579304

In [217]:
function Agent_Location(agent::CHANCE_C.Queue, model::ABM; levee = false, f_e = 0.0, bg_sample_size = 10, house_choice_mode = "simple_anova_utility",
    budget_reduction_perc = 0.10, penalty = 50, migrate_prob = 0.05)
    err_afford = []
    err_better = []
    if agent.type == :relocating
        loc_df = copy(model.df)
        # Create a GEOID-to-BlockGroup lookup
        geoid_to_bg = Dict{Int64, Int64}()
        for bg in allagents(model)
            if bg isa BlockGroup
                geoid_to_bg[bg.GEOID] = bg.id
            end
        end

        # Use view or filter instead of multiple list comprehensions
        moving_agents = sort!([a for a in agents_in_position(agent, model) if a isa HHAgent], by=a -> a.income, rev=true)

        current_index = 1
        #Preallocate some vectors to reduce memory allocations
        hh_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
        bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
        bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
        bg_cat = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
        bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

        for hh_agent in moving_agents
            # Consolidate budget selection logic
            bg_budget = if house_choice_mode == "simple_avoidance_utility"
                hh_agent.avoidance ? 
                    subset(loc_df, :perc_fld_area => n -> n .<= 0.10, view = true) :
                    subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true, view = true)
            elseif house_choice_mode == "budget_reduction"
                new_house_budget = hh_agent.house_budget * (1 - budget_reduction_perc)
                hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, hh_agent.house_budget)
                subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
            else
                subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true, view = true)
            end


            # Use a more efficient sampling approach
            util_diff = 0
            try
                # Precompute weights to avoid repeated calculations
                weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
                    
                # Check for available locations more efficiently
                valid_locations = findall(weights .> 0)
                if isempty(valid_locations)
                    #push!(err_afford, first(keys(hh_agent.utility)))
                    throw(ErrorException("No affordable locations with available units"))
                end

                #Sample from affordable locations based on weights
                sample_size = min(length(valid_locations), bg_sample_size)
                sampled_indices = sample(abmrng(model), valid_locations, sample_size, replace=false)
                
                #Grab utilities from sampled locations
                loc_utilities = [model[geoid_to_bg[row.GEOID]].current_utility[row.income_cat] + ((row.income_cat - hh_agent.group) * penalty) for row in eachrow(bg_budget[sampled_indices, [:GEOID, :income_cat]])]
                # Find indices of block groups with better utilities than current agent location
                current_utility = first(values(hh_agent.utility))
                util_diff = (current_utility - maximum(loc_utilities)) / current_utility
                opt_locs = findall(>(current_utility), loc_utilities)

                # Check if any moves are possible
                if isempty(opt_locs)
                    #push!(err_better, first(keys(hh_agent.utility)))
                    throw(ErrorException("No better locations found"))
                end
                best_indices = sampled_indices[opt_locs]
                
                #Append future block group properties to vectors
                ind_length = length(best_indices)

                copyto!(hh_ids, current_index, fill(hh_agent.id, ind_length), 1, ind_length)
                copyto!(bg_ids, current_index, getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID]), 1, ind_length)
                copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
                copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
                copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)

                current_index += ind_length
                
            catch
                # Migration logic remains similar
                last_bg = model[first(keys(hh_agent.utility))]
                if last_bg.id == -1
                    remove_agent!(hh_agent, model)
                    continue
                end

                stay_prob = 1.5/(1+ exp(-0.6(util_diff)))
                stay_prob = stay_prob <= 1.0 ? stay_prob : 1.0
                if rand(abmrng(model), Binomial(1, stay_prob)) == 1
                    #Revert HHAgent Properties
                    last_util = first(values(hh_agent.utility))
                    setproperty!(hh_agent, :bg_id, last_bg.id)
                    setproperty!(hh_agent, :occ_cat, first([k for (k,v) in last_bg.current_utility if v == last_util]))
                    move_agent!(hh_agent, last_bg.pos, model)
                    #Update Last BG Properties
                    last_bg.occupied_units[hh_agent.occ_cat] += 1
                    last_bg.available_units[hh_agent.occ_cat] -= 1
                    last_bg.population += getproperty(hh_agent, :no_hhs_per_agent) * getproperty(hh_agent, :hh_size)
                else
                    remove_agent!(hh_agent, model)
                end
            end
        end
        
        ##Create df from vectors, append to model properties df
        #Remove extra undef values by using current index
        bg_sample = DataFrame(hh_id = hh_ids[1:current_index-1], bg_id = bg_ids[1:current_index-1], 
        GEOID = bg_GEOID[1:current_index-1], cat = bg_cat[1:current_index-1], bg_utility = bg_utilities[1:current_index-1])
        
        append!(model.hh_utilities_df, bg_sample)
    else
        return 
    end
    #return err_afford, err_better
end

Agent_Location (generic function with 1 method)

In [218]:
CHANCE_C.AgentLocation(phil_abm[0], phil_abm)
#Agent_Location(phil_abm[0], phil_abm)

Row,hh_id,bg_id,GEOID,cat,bg_utility
,Int64,Int64,Int64,Int64,Float64
1,53157,480,421010149003,2,3.36291e5
2,53157,12,421010007003,3,4.11337e5
3,53157,510,421010167011,1,2.79336e5
4,53157,259,421010080004,3,3.95155e5
5,53157,105,421010030021,1,1.81989e5
6,53157,375,421010108004,3,3.26341e5
7,53157,345,421010100003,2,357882.0
8,53157,122,421010033002,3,5.42814e5
9,53157,68,421010022001,2,184969.0


In [219]:
queue_2 = collect(agents_in_position(phil_abm[0], phil_abm))
ag_in_queue_2 = length(queue_2) - 1

594

In [220]:
ag_in_queue_1 - ag_in_queue_2

143

In [221]:
new_pop_1 = length([a for a in allagents(phil_abm) if a isa HHAgent && a.bg_id >= 1])

35439

In [222]:
(new_pop_1+ag_in_queue_2) - (init_pop + no_new_agents)
#(current pop + pop in Queue) - (Orig. pop + migrating agents)

-145

In [171]:
collect(agents_in_position(phil_abm[0], phil_abm))

332-element Vector{AbstractAgent}:
 Main.CHANCE_C.Queue(0, (28, 25), :relocating)
 HHAgent(46282, (28, 25), -1, 10, 2, 0, 2.0, 2, 42496.30234000001, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 56520.08211220001, 0.33)
 HHAgent(47991, (28, 25), -1, 10, 2, 0, 1.0, 1, 58964.47317999999, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 78422.7493294, 0.33)
 HHAgent(48965, (28, 25), -1, 10, 2, 0, 2.0, 3, 39710.902, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, true, 52815.49966000001, 0.33)
 HHAgent(47275, (28, 25), -1, 10, 2, 0, 1.0, 1, 45796.46880000001, Dict(-1 => 0.0), "perc", 0, 0.95, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

In [ ]:
model = phil_abm
market_iter = 1

In [ ]:
moving_agents = [id for id in ids_in_position(model[0], model) if model[id] isa HHAgent]

bg_demand = DataFrame(top_bg = Int64[], top_cat = Int64[], hh_id = Int64[], hh_income = Float64[])

for id in moving_agents
    hh_utilities_subset = model.hh_utilities_df[model.hh_utilities_df.hh_id .== id, :] #Subset hh_utilities_df based on agent choices
    sort!(hh_utilities_subset, :bg_utility, rev=true) #Sort bg candidates from highest to lowest utility
    try
        top_bg = hh_utilities_subset[market_iter, :bg_id] # get the bg name for the top candidate (excluding previous top candidates from previous iterations)
        top_cat = hh_utilities_subset[market_iter, :cat] # get the category name of bg for the top candidate (excluding previous top candidates from previous iterations)
        push!(bg_demand, [top_bg, top_cat, id, model[id].income]) #add bg id, agent id, and agent income to bg_demand
    catch
        #if index is out of range, means agent has gone through all affordable options
        remove_agent!(model[id], model) #remove agent
    end
end

In [ ]:
bg_id, cat = eachrow(unique!(select(bg_demand, [:top_bg, :top_cat])))[1]

bg_subset = bg_demand[(bg_demand.top_bg .== bg_id) .& (bg_demand.top_cat .== cat), :]

if nrow(bg_subset) >= model[bg_id].available_units[cat]
    #subset df further based on available space
    bg_subset = first(sort(bg_subset, :hh_income, rev=true), model[bg_id].available_units[cat])
    model[bg_id].demand_exceeds_supply[cat][model.tick] = true
end

for hh_id in bg_subset.hh_id
    #move agent to bg
    move_agent!(model[hh_id], model[bg_id].pos, model)
    #Update bg_id, housing category location, utility, year of residence of agent
    setproperty!(model[hh_id], :bg_id, bg_id)
    setproperty!(model[hh_id], :occ_cat, cat)
    setproperty!(model[hh_id], :utility, Dict(bg_id => model[bg_id].current_utility[cat]))
    setproperty!(model[hh_id], :year_of_residence, model.tick)
    #update bg attributes
    model[bg_id].occupied_units[cat] += 1
    model[bg_id].available_units[cat] -= 1

    model[bg_id].population += getproperty(model[hh_id],:no_hhs_per_agent) * getproperty(model[hh_id],:hh_size)
end




In [ ]:
model[bg_id].available_units
#eachrow(unique!(select(bg_demand, [:top_bg, :top_cat])))

In [223]:
Housing_Market(phil_abm)

In [224]:
new_pop_2 = length([a for a in allagents(phil_abm) if a isa HHAgent && a.bg_id >= 1])

35849

In [225]:
(new_pop_1+ag_in_queue_2) - new_pop_2

184

In [43]:
331+408

739

In [57]:
function Housing_Market(model::ABM; market_mode = "top_candidate", bg_sample_size = 10) #start with just relocating Queue
    for market_iter in 1:bg_sample_size
        moving_agents = [id for id in ids_in_position(model[0], model) if model[id] isa HHAgent]
        #Check to see if relocating queue is empty
        if length(moving_agents) < 1
            break
        end
        bg_demand = DataFrame(top_bg = Int64[], top_cat = Int64[], hh_id = Int64[], hh_income = Float64[])

        for id in moving_agents
            hh_utilities_subset = model.hh_utilities_df[model.hh_utilities_df.hh_id .== id, :] #Subset hh_utilities_df based on agent choices
            sort!(hh_utilities_subset, :bg_utility, rev=true) #Sort bg candidates from highest to lowest utility
            try
                top_bg = hh_utilities_subset[market_iter, :bg_id] # get the bg name for the top candidate (excluding previous top candidates from previous iterations)
                top_cat = hh_utilities_subset[market_iter, :cat] # get the category name of bg for the top candidate (excluding previous top candidates from previous iterations)
                push!(bg_demand, [top_bg, top_cat, id, model[id].income]) #add bg id, agent id, and agent income to bg_demand
            catch
                #if index is out of range, means agent has gone through all affordable options
                #remove_agent!(model[id], model) #remove agent
            end
        end
        #Move agents to desired bg, if possible 
        for (bg_id, cat) in eachrow(unique!(select(bg_demand, [:top_bg, :top_cat])))
            bg_subset = bg_demand[(bg_demand.top_bg .== bg_id) .& (bg_demand.top_cat .== cat), :]
            if nrow(bg_subset) >= model[bg_id].available_units[cat]
                #subset df further based on available space
                bg_subset = first(sort(bg_subset, :hh_income, rev=true), model[bg_id].available_units[cat])
                model[bg_id].demand_exceeds_supply[cat][model.tick] = true
            end

            for hh_id in bg_subset.hh_id
                #move agent to bg
                move_agent!(model[hh_id], model[bg_id].pos, model)
                #Update bg_id, utility,  year of residence of agent
                setproperty!(model[hh_id], :bg_id, bg_id)
                setproperty!(model[hh_id], :occ_cat, cat)
                setproperty!(model[hh_id], :utility, Dict(bg_id => model[bg_id].current_utility[cat]))
                setproperty!(model[hh_id], :year_of_residence, model.tick)
                #update bg attributes
                model[bg_id].occupied_units[cat] += 1
                model[bg_id].available_units[cat] -= 1

                model[bg_id].population += getproperty(model[hh_id],:no_hhs_per_agent) * getproperty(model[hh_id],:hh_size)
            end
            
        end

    end

    #for any households remaining in queues, assume they migrate
    #remove_agent!.([a for a in agents_in_position(model[0], model) if a isa HHAgent], Ref(model))
end

Housing_Market (generic function with 1 method)

In [ ]:
##Extract pop characteristics from pop_df
pop_df = copy(phil_cbsa_base_pop)
group_col = "adj_income_2019"
cutoffs = OrderedDict("low"=> [0,25000.00], "medium"=>[25000.00,75000.00], "high"=>[75000.00, 1e7])
no_hhs_per_agent = 10
#Subset to only occupied households
pop_hh_df = subset(pop_df, :NP => x -> x .> 0.0)

#Create group labels by group col
pop_hh_df[:, :category] = cut(pop_hh_df[:, group_col], unique(reduce(vcat, collect(values(cutoffs)))), labels = collect(keys(cutoffs)))
#groupby category column 
pop_cat_df = groupby(pop_hh_df, :category)

#Create empty DataFrame
agent_df = DataFrame(nrow = Int64[], cat = String[], race = Float64[], avg_hh_size = Float64[], avg_income = Float64[])
for (i,sub_df) in enumerate(pop_cat_df)
    sort!(sub_df, :adj_income_2019)
    sub_df[:,:group] = map(x->div(x,no_hhs_per_agent), 1:nrow(sub_df))
    hh_bins = combine(groupby(sub_df, :group), nrow, :category => maximum => :cat, :RAC1P => (r -> mode(r)) => :race,  [:NP, :adj_income_2019] .=> mean .=> [:avg_hh_size, :avg_income])
    inc_w = ProbabilityWeights(hh_bins.avg_income ./ sum(hh_bins.avg_income)) #Calculate weights based on avg income
    append!(agent_df, hh_bins[sample(abmrng(phil_abm), 1:nrow(hh_bins), inc_w, migrant_cat_pop[i]; replace = true),2:end]) #Sample rows based on migrant category count
end

In [ ]:
agent_df

In [ ]:
t_g = groupby(agent_df, :cat)[1]
inc_w = ProbabilityWeights(t_g.avg_income ./ sum(t_g.avg_income))
t_g[sample(abmrng(phil_abm), 1:nrow(t_g), inc_w, migrant_cat_pop[1]; replace = true), :]

In [ ]:
### Test model functions
test_bg = phil_abm[10]
println("Occupied: ",test_bg.occupied_units)
println("Available: ",test_bg.available_units)
println("Population: ",test_bg.population)

In [ ]:
length([a for a in agents_in_position(test_bg, phil_abm) if a isa HHAgent && a.group == "medium"])

In [ ]:
CHANCE_C.agent_prob!(test_bg, phil_abm)

In [ ]:
println("Occupied: ",test_bg.occupied_units)
println("Available: ",test_bg.available_units)
println("Population: ",test_bg.population)

In [ ]:
collect(agents_in_position(phil_abm[0].pos, phil_abm))

In [ ]:
function agent_locate(agent::CHANCE_C.Queue, model::ABM; levee = false, f_e = 0.0, bg_sample_size = 10, house_choice_mode = "simple_anova_utility",
    budget_reduction_perc = 0.10, penalty = 50, migrate_prob = 0.05)
    
    loc_df = copy(model.df)
    # Preallocate the DataFrame with a reasonable initial capacity
    #bg_sample = DataFrame(hh_id = Int64[], bg_id = Int64[], GEOID = Int64[], cat = String[], bg_utility = Float64[])

    # Use view or filter instead of multiple list comprehensions
    moving_agents = sort!([a for a in agents_in_position(agent, model) if a isa HHAgent], by=a -> a.income, rev=true)

    current_index = 1
    #Preallocate some vectors to reduce memory allocations
    hh_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
    bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
    bg_cat = Vector{String}(undef, bg_sample_size* length(moving_agents))
    bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

    for hh_agent in moving_agents
        # Consolidate budget selection logic
        bg_budget = if house_choice_mode == "simple_avoidance_utility"
            hh_agent.avoidance ? 
                subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
                subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true)
        elseif house_choice_mode == "budget_reduction"
            new_house_budget = hh_agent.house_budget * (1 - budget_reduction_perc)
            hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, hh_agent.house_budget)
            subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
        else
            subset(loc_df, :market_value => n -> n .<= hh_agent.house_budget, skipmissing=true, view = true)
        end

        # Use a more efficient sampling approach
        try
            
            # Precompute weights to avoid repeated calculations
            weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
            
            # Check for available locations more efficiently
            valid_locations = findall(weights .> 0)
            if isempty(valid_locations)
                throw(ErrorException("No affordable locations with available units"))
            end

            #Sample from affordable locations based on weights
            sample_size = min(length(valid_locations), bg_sample_size)
            sampled_indices = sample(abmrng(model), valid_locations, sample_size, replace=false)
    
            #Grab utilities from sampled locations
            bg_sel = Iterators.filter(bg -> bg isa BlockGroup && bg.GEOID in bg_budget[sampled_indices, :GEOID], allagents(model)).id
            loc_utilities = getindex.(getproperty.(getindex.(Ref(model), bg_sel), :current_utility), bg_budget[sampled_indices, :income_cat])
            # Find indices of block groups with better utilities than current agent location
            current_utility = first(values(hh_agent.utility))
            opt_locs = findall(>(current_utility), loc_utilities)

            # Check if any moves are possible
            if isempty(opt_locs)
                throw(ErrorException("No better locations found"))
            end
            best_indices = sampled_indices[opt_locs]

            #Append future block group properties to vectors
            ind_length = length(best_indices)

            copyto!(hh_ids, current_index, fill(hh_agent.id, ind_length), 1, ind_length)
            copyto!(bg_ids, current_index, collect(bg_sel)[opt_locs], 1, ind_length)
            copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
            copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
            copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)

            current_index += ind_length

        catch
            # Migration logic remains similar
            if rand(abmrng(model), Binomial(1, migrate_prob)) == 1
                last_bg = model[first(keys(hh_agent.utility))]
                move_agent!(hh_agent, last_bg.pos, model)
                last_bg.occupied_units[hh_agent.group] += 1
                last_bg.available_units[hh_agent.group] -= 1
                last_bg.population += getproperty(hh_agent, :no_hhs_per_agent) * getproperty(hh_agent, :hh_size)
            else
                remove_agent!(hh_agent, model)
            end
        end
    end
    
    ##Create df from vectors, append to model properties df
    #Remove extra undef values by using current index
    bg_sample = DataFrame(hh_id = hh_ids[1:current_index-1], bg_id = bg_ids[1:current_index-1], 
    GEOID = bg_GEOID[1:current_index-1], cat = bg_cat[1:current_index-1], bg_utility = bg_utilities[1:current_index-1])
    
    append!(model.hh_utilities_df, bg_sample)
end

In [ ]:
agent_locate(phil_abm[0], phil_abm)


In [ ]:
phil_abm.hh_utilities_df

In [ ]:
#Breakdown agent relocation function:
loc_df = copy(phil_abm.df)
#Create a GEOID-to-BlockGroup lookup
geoid_to_bg = Dict{Int64, Int64}()
for bg in allagents(phil_abm)
    if bg isa BlockGroup
        geoid_to_bg[bg.GEOID] = bg.id
    end
end
house_choice_mode = "simple_anova_utility"
bg_sample_size = 10
# Use view or filter instead of multiple list comprehensions
moving_agents = sort!([a for a in agents_in_position(phil_abm[0], phil_abm) if a isa CHANCE_C.HHAgent], by=a -> a.income, rev=true)
agent_sel = moving_agents[1]
current_index = 1
# Preallocate some vectors to reduce memory allocations
bg_ids = Vector{Int64}(undef, bg_sample_size * length(moving_agents))
bg_GEOID = Vector{Int64}(undef, bg_sample_size* length(moving_agents))
bg_cat = Vector{String}(undef, bg_sample_size* length(moving_agents))
bg_utilities = Vector{Float64}(undef, bg_sample_size * length(moving_agents))

# Consolidate budget selection logic
bg_budget = if house_choice_mode == "simple_avoidance_utility"
    agent_sel.avoidance ? 
        subset(loc_df, :perc_fld_area => n -> n .<= 0.10) :
        subset(loc_df, :market_value => n -> n .<= agent_sel.house_budget, skipmissing=true)
elseif house_choice_mode == "budget_reduction"
    new_house_budget = agent_sel.house_budget * (1 - budget_reduction_perc)
    hh_budget = ifelse.(loc_df.perc_fld_area .>= 0.10, new_house_budget, agent_sel.house_budget)
    subset(loc_df, :market_value => n -> n .<= hh_budget, skipmissing=true, view = true)
else
    subset(loc_df, :market_value => n -> n .<= agent_sel.house_budget, skipmissing=true, view = true)
end




In [ ]:
# Use a more efficient sampling approach
#try
    # Precompute weights to avoid repeated calculations
weights = ProbabilityWeights(bg_budget.available_units ./ sum(bg_budget.available_units))
            
    # Check for available locations more efficiently
valid_locations = findall(weights .> 0)
if isempty(valid_locations)
    throw(ErrorException("No affordable locations with available units"))
end
    # Efficient sampling
sample_size = min(length(valid_locations), bg_sample_size)
sampled_indices = sample(abmrng(phil_abm), valid_locations, sample_size, replace=false)
    #bg_options = bg_budget[sampled_indices, :]
    
    #Grab utilities from sampled locations
#bg_sel = map(bg -> bg.id, Iterators.filter(bg -> bg isa BlockGroup && bg.GEOID in bg_budget[sampled_indices, :GEOID], allagents(phil_abm)))
loc_utilities = [phil_abm[geoid_to_bg[row.GEOID]].current_utility[row.income_cat] for row in eachrow(bg_budget[sampled_indices, [:GEOID, :income_cat]])]
# Find indices of block groups with better utilities than current agent location
current_utility = first(values(agent_sel.utility))
opt_locs = findall(>(current_utility), loc_utilities)
# Check if any moves are possible
if isempty(opt_locs)
    throw(ErrorException("No better locations found"))
end
best_indices = sampled_indices[opt_locs]
#catch
#    println("didnt work!")
#end

In [ ]:
println(sampled_indices)
println(opt_locs)
println(best_indices)


In [ ]:
getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID])

In [ ]:
ind_length = length(best_indices)   
#bg_ids = getproperty.(bg_sel[opt_locs], :id)
#bg_GEOID = bg_budget[sampled_indices, :GEOID]
#bg_cat = bg_budget[sampled_indices, :income_cat]
#bg_utilities = loc_utilities[opt_locs]

copyto!(bg_ids, current_index, getindex.(Ref(geoid_to_bg), bg_budget[best_indices,:GEOID]), 1, ind_length)
copyto!(bg_GEOID, current_index, bg_budget[best_indices, :GEOID], 1, ind_length)
copyto!(bg_cat, current_index, bg_budget[best_indices, :income_cat], 1, ind_length)
copyto!(bg_utilities, current_index, loc_utilities[opt_locs], 1, ind_length)
current_index += ind_length


# Append to bg_sample
#append!(bg_sample, move_df)

In [ ]:
fill(agent_sel.id, ind_length)

In [ ]:
agent_locate(phil_abm[0], phil_abm)

In [ ]:
agent_relocate(phil_abm[0], phil_abm)

In [ ]:
sort!(filter(a -> a isa HHAgent, agents_in_position(phil_abm[0], phil_abm)), by=a -> a.income, rev=true)

In [ ]:
step!(phil_abm)

In [ ]:
phil_abm.tick